# Classifying Penguins with Keras

In [86]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn import preprocessing
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

In [87]:
! pip install palmerpenguins
from palmerpenguins import load_penguins
penguins = load_penguins()
penguins.head()


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007


In [88]:
#drop nan rows
penguins = penguins.dropna()
penguins.shape

(333, 8)

In [89]:
#shuffle the data
penguins = penguins.sample(frac=1, random_state = 42).reset_index(drop=True)

In [90]:
penguins_x = pd.concat([penguins[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']], pd.get_dummies(penguins['sex'])], axis = 1)
# penguins_x = penguins_x[['body_mass_g', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'female', 'male']]
penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,3250.0,39.5,16.7,178.0,True,False
1,3675.0,50.9,17.9,196.0,True,False
2,4000.0,42.1,19.1,195.0,False,True
3,4850.0,46.6,14.2,210.0,True,False
4,4050.0,41.1,18.2,192.0,False,True
...,...,...,...,...,...,...
328,4750.0,49.6,15.0,216.0,False,True
329,3900.0,37.2,19.4,184.0,False,True
330,3200.0,39.7,17.7,193.0,True,False
331,3950.0,45.2,17.8,198.0,True,False


In [91]:
x = penguins_x.values
min_max_scaler = preprocessing.MinMaxScaler()
scaled_penguins_x = pd.DataFrame(min_max_scaler.fit_transform(x), columns=penguins_x.columns)
scaled_penguins_x

,body_mass_g,bill_length_mm,bill_depth_mm,flipper_length_mm,female,male
0,0.152778,0.269091,0.428571,0.101695,1.0,0.0
1,0.270833,0.683636,0.571429,0.406780,1.0,0.0
2,0.361111,0.363636,0.714286,0.389831,0.0,1.0
3,0.597222,0.527273,0.130952,0.644068,1.0,0.0
4,0.375000,0.327273,0.607143,0.338983,0.0,1.0
...,...,...,...,...,...,...
328,0.569444,0.636364,0.226190,0.745763,0.0,1.0
329,0.333333,0.185455,0.750000,0.203390,0.0,1.0
330,0.138889,0.276364,0.547619,0.355932,1.0,0.0
331,0.347222,0.476364,0.559524,0.440678,1.0,0.0


In [92]:
penguins_y = penguins['species']
print(penguins_y)
penguins_y = penguins_y.astype('category').cat.codes.to_numpy()
penguins_y

0         Adelie
1      Chinstrap
2         Adelie
3         Gentoo
4         Adelie
         ...    
328       Gentoo
329       Adelie
330       Adelie
331    Chinstrap
332       Adelie
Name: species, Length: 333, dtype: object


array([0, 1, 0, 2, 0, 1, 1, 2, 2, 2, 0, 0, 1, 0, 1, 0, 0, 2, 0, 1, 0, 0, 1, 2, 0, 0, 2, 1, 2, 1, 2, 1, 0, 0, 1, 1, 2, 2, 0, 0, 0,
       0, 2, 2, 0, 0, 1, 0, 0, 1, 0, 2, 2, 0, 0, 2, 0, 0, 2, 2, 1, 1, 1, 0, 0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 2, 2, 0, 0, 0, 2, 0, 0,
       2, 0, 1, 2, 0, 1, 2, 2, 2, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 1, 0, 2, 0, 2, 2, 0, 2, 0, 1, 0, 2, 2, 2, 0, 2, 0, 2, 0, 2, 1,
       0, 0, 1, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 2, 0, 1, 0, 0, 2, 0, 1, 2, 1, 2, 1, 2, 2, 2, 2, 0, 0, 2, 2, 2, 0, 2, 2, 0, 1, 1, 1,
       2, 2, 2, 2, 2, 0, 0, 2, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 2, 1, 0, 2, 2, 0, 1, 0, 1, 0, 2, 0, 2, 0, 0, 0, 2, 0, 2, 0, 1, 0, 0,
       2, 2, 2, 0, 0, 0, 2, 2, 0, 0, 1, 0, 2, 0, 1, 1, 1, 0, 2, 1, 2, 2, 0, 2, 0, 0, 2, 0, 2, 0, 2, 1, 0, 1, 2, 1, 0, 2, 2, 2, 0,
       0, 0, 2, 2, 2, 1, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 1, 1, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 2, 2, 0, 2, 2, 2, 0, 0, 0, 0, 0, 2,
       0, 2, 0, 0, 2, 2, 0, 0, 1, 2, 1, 0, 1, 2, 0, 0, 2, 2, 1, 2, 2, 0, 0, 2, 2, 0, 2, 1,

In [93]:
#construct the model
inputs = keras.Input(shape=(6,))
x = layers.Dense(7, activation = 'relu')(inputs)
x = layers.Dense(5, activation = 'relu')(x)
x = layers.Dense(3, activation = 'relu')(x)
#make sure the model activation is right for 3 categories, not sigmoid
outputs = layers.Dense(3, activation='softmax')(x)
model = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model")

In [94]:
model.summary()

Model: "penguin_model"
┌──────────────────────────────────────┬─────────────────────────────┬─────────────────┐
│ Layer (type)                         │ Output Shape                │         Param # │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ input_layer_6 (InputLayer)           │ (None, 6)                   │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_24 (Dense)                     │ (None, 7)                   │              49 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_25 (Dense)                     │ (None, 5)                   │              40 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_26 (Dense)                     │ (None, 3)                   │              18 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────

In [95]:
#don't have the right library to run, don't worry about it
keras.utils.plot_model(model, show_shapes = True)

You must install pydot (`pip install pydot`) for `plot_model` to work.


In [96]:
#issue with this code, nan fields in penguins datasets, so need to add code cell to get rid of NAN's 
#but for now dropping rows with nan's, see above
model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)
#make sure to shuffle the data
history = model.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs=100, validation_split=0.1)

scores = model.evaluate(scaled_penguins_x, penguins_y, verbose=2)

Epoch 1/100


C:\Users\karis\anaconda3\Lib\site-packages\keras\src\backend\tensorflow\nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.3512 - loss: 1.0799 - val_accuracy: 0.4118 - val_loss: 1.0628
Epoch 2/100
1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 0.3906 - loss: 1.0597

C:\Users\karis\anaconda3\Lib\site-packages\keras\src\backend\tensorflow\nn.py:1216: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.3512 - loss: 1.0712 - val_accuracy: 0.4118 - val_loss: 1.0571
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3512 - loss: 1.0653 - val_accuracy: 0.4118 - val_loss: 1.0523
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.3545 - loss: 1.0605 - val_accuracy: 0.4118 - val_loss: 1.0482
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3645 - loss: 1.0563 - val_accuracy: 0.4118 - val_loss: 1.0442
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3645 - loss: 1.0519 - val_accuracy: 0.4118 - val_loss: 1.0402
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.3712 - loss: 1.0478 - val_accuracy: 0.4706 - val_loss: 1.0366
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.3846 - loss: 1.0436 - val_accuracy: 0.4706 - val_loss: 1.0328
Epoch 9/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3946 - loss: 1.0394 - val_accuracy: 0.5000 - val_loss: 1.0291
Epoc

In [97]:
model_logit_false = keras.Model(inputs=inputs, outputs=outputs, name="penguin_model_scaled")

model_logit_false.compile(
    #don't recalculate the classes, already told the model the info it needs with softmax
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    optimizer=keras.optimizers.RMSprop(),
    metrics=["accuracy"],
)

history_logit_true = model_logit_false.fit(scaled_penguins_x, penguins_y, batch_size = 64, epochs = 100, validation_split = 0.1)

scores = model_logit_false.evaluate(scaled_penguins_x, penguins_y, verbose = 2)

Epoch 1/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 1s 83ms/step - accuracy: 0.7960 - loss: 0.4806 - val_accuracy: 0.7941 - val_loss: 0.4647
Epoch 2/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7960 - loss: 0.4734 - val_accuracy: 0.7941 - val_loss: 0.4586
Epoch 3/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - accuracy: 0.7960 - loss: 0.4685 - val_accuracy: 0.7941 - val_loss: 0.4533
Epoch 4/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.7960 - loss: 0.4643 - val_accuracy: 0.7941 - val_loss: 0.4490
Epoch 5/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7960 - loss: 0.4606 - val_accuracy: 0.7941 - val_loss: 0.4452
Epoch 6/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.7960 - loss: 0.4572 - val_accuracy: 0.7941 - val_loss: 0.4408
Epoch 7/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7960 - loss: 0.4539 - val_accuracy: 0.7941 - val_loss: 0.4378
Epoch 8/100
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7960 - loss: 0.4507 - val_accuracy: 0.7941 - val_loss:

In [98]:
model_logit_false.predict(scaled_penguins_x)

11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 


array([[8.95105839e-01, 1.04887411e-01, 6.70874897e-06],
       [7.06314296e-02, 8.99139643e-01, 3.02289203e-02],
       [7.86704183e-01, 2.13157609e-01, 1.38232921e-04],
       [1.04339642e-06, 1.78738753e-03, 9.98211503e-01],
       [8.23158860e-01, 1.76732257e-01, 1.08799322e-04],
       [2.44996950e-01, 7.50822902e-01, 4.18018224e-03],
       [6.14204228e-01, 3.85446638e-01, 3.49090464e-04],
       [6.37400763e-06, 3.96108022e-03, 9.96032536e-01],
       [1.93274978e-08, 1.07932981e-04, 9.99891996e-01],
       [3.04788027e-05, 1.44197242e-02, 9.85549867e-01],
       [4.80771065e-01, 5.15277922e-01, 3.95102520e-03],
       [8.94829810e-01, 1.05145164e-01, 2.51023303e-05],
       [6.78352594e-01, 3.21529090e-01, 1.18270800e-04],
       [8.19864511e-01, 1.80112258e-01, 2.31468493e-05],
       [3.20538044e-01, 6.77724063e-01, 1.73791882e-03],
       [8.46126974e-01, 1.53848246e-01, 2.47191037e-05],
       [8.96443129e-01, 1.03549331e-01, 7.59658269e-06],
       [1.22250322e-06, 1.77134

In [99]:
penguins['species']

0         Adelie
1      Chinstrap
2         Adelie
3         Gentoo
4         Adelie
         ...    
328       Gentoo
329       Adelie
330       Adelie
331    Chinstrap
332       Adelie
Name: species, Length: 333, dtype: object

In [100]:
penguins_y

array([0, 1, 0, 2, 0, 1, 1, 2, 2, 2, 0, 0, 1, 0, 1, 0, 0, 2, 0, 1, 0, 0, 1, 2, 0, 0, 2, 1, 2, 1, 2, 1, 0, 0, 1, 1, 2, 2, 0, 0, 0,
       0, 2, 2, 0, 0, 1, 0, 0, 1, 0, 2, 2, 0, 0, 2, 0, 0, 2, 2, 1, 1, 1, 0, 0, 1, 0, 2, 0, 1, 0, 0, 2, 1, 2, 2, 0, 0, 0, 2, 0, 0,
       2, 0, 1, 2, 0, 1, 2, 2, 2, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 1, 1, 0, 2, 0, 2, 2, 0, 2, 0, 1, 0, 2, 2, 2, 0, 2, 0, 2, 0, 2, 1,
       0, 0, 1, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 2, 0, 1, 0, 0, 2, 0, 1, 2, 1, 2, 1, 2, 2, 2, 2, 0, 0, 2, 2, 2, 0, 2, 2, 0, 1, 1, 1,
       2, 2, 2, 2, 2, 0, 0, 2, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 2, 1, 0, 2, 2, 0, 1, 0, 1, 0, 2, 0, 2, 0, 0, 0, 2, 0, 2, 0, 1, 0, 0,
       2, 2, 2, 0, 0, 0, 2, 2, 0, 0, 1, 0, 2, 0, 1, 1, 1, 0, 2, 1, 2, 2, 0, 2, 0, 0, 2, 0, 2, 0, 2, 1, 0, 1, 2, 1, 0, 2, 2, 2, 0,
       0, 0, 2, 2, 2, 1, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 1, 1, 2, 1, 2, 2, 2, 1, 2, 1, 1, 1, 2, 2, 0, 2, 2, 2, 0, 0, 0, 0, 0, 2,
       0, 2, 0, 0, 2, 2, 0, 0, 1, 2, 1, 0, 1, 2, 0, 0, 2, 2, 1, 2, 2, 0, 0, 2, 2, 0, 2, 1,